In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('../data/telco_cleaned.csv')
df.shape

(7043, 25)

In [14]:
pd.set_option('display.max_columns', None)
df.head(5)

,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Paperless Billing,Monthly Charges,Total Charges,Churn Value,CLTV,Internet Service_Fiber optic,Internet Service_No,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,0,0,0,2,1,0,1,1,0,0,0,0,1,53.85,108.15,1,3239,False,False,False,False,False,False,True
1,0,0,1,2,1,0,0,0,0,0,0,0,1,70.70,151.65,1,2701,True,False,False,False,False,True,False
2,0,0,1,8,1,1,0,0,1,0,1,1,1,99.65,820.50,1,5372,True,False,False,False,False,True,False
3,0,1,1,28,1,1,0,0,1,1,1,1,1,104.80,3046.05,1,5003,True,False,False,False,False,True,False
4,0,0,1,49,1,1,0,1,1,0,1,1,1,103.70,5036.30,1,5340,True,False,False,False,False,False,False


In [4]:
correlations = df.corr(numeric_only=True)['CLTV'].sort_values(ascending=False)
correlations

CLTV                                      1.000000
Tenure Months                             0.396406
Total Charges                             0.342091
Contract_Two year                         0.233151
Online Backup                             0.138610
Online Security                           0.138354
Partner                                   0.137859
Multiple Lines                            0.133568
Device Protection                         0.120394
Tech Support                              0.119132
Streaming Movies                          0.113000
Streaming TV                              0.104889
Monthly Charges                           0.098693
Payment Method_Credit card (automatic)    0.078995
Dependents                                0.058222
Contract_One year                         0.056364
Phone Service                             0.023015
Paperless Billing                         0.011501
Internet Service_Fiber optic              0.011159
Senior Citizen                 

In [5]:
df = df.drop(columns=['Churn Label'])

X = df.drop(columns=['CLTV', 'Churn Value'])
y = df['CLTV']

X.shape, y.shape

((7043, 22), (7043,))

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((5634, 22), (1409, 22))

In [7]:
numeric_features = ['Monthly Charges', 'Total Charges', 'Tenure Months']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

X_train_scaled[numeric_features].describe()

,Monthly Charges,Total Charges,Tenure Months
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-8.828185e-17,5.044677e-17,-6.873372e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.529008e+00,-1.005361e+00,-1.321126e+00
25%,-9.843217e-01,-8.292801e-01,-9.545329e-01
50%,1.911405e-01,-3.933310e-01,-1.398817e-01
75%,8.417837e-01,6.612306e-01,9.191649e-01
max,1.793743e+00,2.831610e+00,1.611618e+00


In [8]:
X_train_scaled = X_train_scaled.drop(columns=['Total Charges'])
X_test_scaled = X_test_scaled.drop(columns=['Total Charges'])

X_train_scaled.shape, X_test_scaled.shape

((5634, 21), (1409, 21))

In [9]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"RMSE: {rmse_lr:.2f}")
print(f"MAE: {mae_lr:.2f}")
print(f"R²: {r2_lr:.4f}")

RMSE: 1062.81
MAE: 897.00
R²: 0.1685


In [10]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly = lr_poly.predict(X_test_poly)

rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
mae_poly = mean_absolute_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print(f"RMSE: {rmse_poly:.2f}")
print(f"MAE: {mae_poly:.2f}")
print(f"R²: {r2_poly:.4f}")

RMSE: 1064.00
MAE: 900.70
R²: 0.1666


In [11]:
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

for name, y_pred in [('Lasso', y_pred_lasso), ('Ridge', y_pred_ridge)]:
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")

Lasso -> RMSE: 1062.26, MAE: 897.14, R²: 0.1694
Ridge -> RMSE: 1062.66, MAE: 897.00, R²: 0.1687


In [12]:
df_raw = pd.read_excel("../data/Telco_customer_churn.xlsx")
churn_score = df_raw['Churn Score']

X_train_with_score = X_train_scaled.copy()
X_test_with_score = X_test_scaled.copy()
X_train_with_score['Churn Score'] = churn_score.loc[X_train.index].values
X_test_with_score['Churn Score'] = churn_score.loc[X_test.index].values

lr2 = LinearRegression()
lr2.fit(X_train_with_score, y_train)
y_pred2 = lr2.predict(X_test_with_score)

print("R² with Churn Score:", r2_score(y_test, y_pred2))
print("R² without (baseline):", r2_lr)

R² with Churn Score: 0.1686315860346007
R² without (baseline): 0.16848949660135992


In [13]:
models = {
    'Linear': lr,
    'Polynomial': lr_poly,
    'Lasso': lasso,
    'Ridge': ridge
}

for name, model in models.items():
    if name == 'Polynomial':
        train_r2 = r2_score(y_train, model.predict(X_train_poly))
        test_r2 = r2_score(y_test, model.predict(X_test_poly))
    else:
        train_r2 = r2_score(y_train, model.predict(X_train_scaled))
        test_r2 = r2_score(y_test, model.predict(X_test_scaled))
    print(f"{name:12s} Train R²: {train_r2:.4f}   Test R²: {test_r2:.4f}")

Linear       Train R²: 0.1573   Test R²: 0.1685
Polynomial   Train R²: 0.2117   Test R²: 0.1666
Lasso        Train R²: 0.1569   Test R²: 0.1694
Ridge        Train R²: 0.1573   Test R²: 0.1687


overfitting is clear since the r squared in training data is completely different than the test data